# Financial Fraud Detection — Exploratory Data Analysis

This notebook explores the Credit Card Fraud Detection dataset before modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Same colours and clean style as the Streamlit app
TEAL, CORAL, NAVY = "#2A9D8F", "#E76F51", "#1F2A44"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e8ebf2", "axes.axisbelow": True})

df = pd.read_csv("../data/creditcard.csv")
df.shape

## 1. Basic dataset info

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Column names:", list(df.columns))
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum().sum())
print("Duplicate records:", df.duplicated().sum())

In [ ]:
df.describe()

## 2. Class balance

In [ ]:
counts = df["Class"].value_counts().sort_index()
fraud_pct = counts[1] / counts.sum() * 100
print("Genuine:", counts[0])
print("Fraud:", counts[1])
print(f"Fraud percentage: {fraud_pct:.4f}%")

plt.figure()
plt.bar(["Genuine", "Fraud"], [counts[0], counts[1]], color=["#2A9D8F", "#E76F51"])
plt.ylabel("Number of Transactions")
plt.title("Genuine vs Fraudulent Transactions")
plt.show()

## 3. Transaction amount distribution

In [ ]:
plt.figure()
plt.hist(df["Amount"], bins=50, color="#2A9D8F")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.title("Transaction Amount Distribution")
plt.show()

## 4. Fraudulent transaction amount distribution

In [ ]:
fraud_amounts = df[df["Class"] == 1]["Amount"]
plt.figure()
plt.hist(fraud_amounts, bins=50, color="#E76F51")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.title("Fraudulent Transaction Amount Distribution")
plt.show()

## 5. Transaction time distribution

In [ ]:
plt.figure()
plt.hist(df["Time"], bins=50, color="#1F2A44")
plt.xlabel("Time (seconds since first transaction)")
plt.ylabel("Frequency")
plt.title("Transaction Time Distribution")
plt.show()

## 6. Correlation analysis

In [ ]:
corr = df.corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.title("Feature Correlation Heatmap")
plt.show()

# Features most correlated with the target class
corr["Class"].abs().sort_values(ascending=False).head(10)

## 7. Fraud vs genuine: amounts compared

Fraud is not just "big transactions", so amount alone cannot separate the classes.

In [ ]:
print(df.groupby("Class")["Amount"].describe().round(2))

plt.figure(figsize=(6, 3.5))
plt.boxplot([df[df.Class == 0]["Amount"], df[df.Class == 1]["Amount"]],
            labels=["Genuine", "Fraud"], showfliers=False)
plt.ylabel("Amount (outliers hidden)")
plt.title("Amount by class")
plt.show()

## 8. Why accuracy is misleading here

A model that predicts "genuine" for everything scores about 99.8% accuracy and catches **zero** fraud.

In [ ]:
baseline_accuracy = (df["Class"] == 0).mean()
print(f"Accuracy of always predicting genuine: {baseline_accuracy:.4%}")
print("Fraud caught by that model: 0")

## Notes

- `V1`–`V28` are anonymized, PCA-transformed features and do not have
  real-world interpretations (e.g. they are **not** "age" or "income").
- The dataset is highly imbalanced (~0.17% fraud), which motivates the
  use of SMOTE and imbalance-aware metrics (Precision, Recall, F1,
  PR-AUC) in the modeling stage — see `train.py`.

## Conclusions

1. Fraud is extremely rare, so accuracy is useless and recall / precision / PR-AUC matter.
2. A handful of V-features correlate noticeably with fraud, which tree models can exploit.
3. Amount and Time alone do not separate fraud from genuine, so a trained model is needed.
4. Next stage (see `train.py`): split, scale, SMOTE, train three models, compare, deploy the best.